In [1]:
try:
    from datasets import load_dataset
except ImportError:
    %pip install datasets
    from datasets import load_dataset

train_dataset = load_dataset("Francesco/pills-sxdht", split="train")
valid_dataset = load_dataset("Francesco/pills-sxdht", split="validation")
test_dataset  = load_dataset("Francesco/pills-sxdht", split="test")

In [2]:
import torch
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT  # pesos preentrenados en COCO
model = fasterrcnn_resnet50_fpn_v2(weights=weights)
model.to(device)
model.eval()

preprocess = weights.transforms()  # transformaciones asociadas a los pesos
print(f"Modelo cargado en {device}")

Modelo cargado en cpu


In [6]:
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# El ClassLabel del dataset trae 9 nombres, pero solo 7 son clases reales:
#   0 'pills'          -> supercategoría de Roboflow, sin ninguna anotación
#   2 'Ibuphil 600 mg' -> solo 1 caja en train y 1 en test (descartada)
all_names = train_dataset.features["objects"]["category"].feature.names
print(f"Todos los nombres de clases {len(all_names)} : {all_names}")
KEEP_IDS = [1, 3, 4, 5, 6, 7, 8]
DROP_IDS = {0, 2}

# id original -> etiqueta del modelo (1..7); el 0 queda reservado para el fondo
ID2LABEL = {orig: new for new, orig in enumerate(KEEP_IDS, start=1)}
CLASSES = ["__background__"] + [all_names[i] for i in KEEP_IDS]
print("Clases:", CLASSES[1:])

num_classes = len(CLASSES)  # 7 + fondo = 8

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
model.to(device)
print(f"Nueva cabeza con {num_classes} clases (incluido fondo)")

Todos los nombres de clases 9 : ['pills', 'Cipro 500', 'Ibuphil 600 mg', 'Ibuphil Cold 400-60', 'Xyzall 5mg', 'blue', 'pink', 'red', 'white']
Clases: ['Cipro 500', 'Ibuphil Cold 400-60', 'Xyzall 5mg', 'blue', 'pink', 'red', 'white']
Nueva cabeza con 8 clases (incluido fondo)


In [4]:
from torch.utils.data import Dataset, DataLoader
from torchvision import tv_tensors
from torchvision.transforms import v2


class PillsDataset(Dataset):
    """Adapta el dataset de HF al formato que espera Faster R-CNN de torchvision."""

    def __init__(self, hf_ds, transforms=None):
        # Se descartan las imágenes que contienen clases eliminadas,
        # para no dejar objetos visibles sin anotar (el modelo los aprendería como fondo)
        self.ds = hf_ds.filter(lambda o: not any(c in DROP_IDS for c in o["category"]),
                               input_columns="objects")
        self.transforms = transforms

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        img = item["image"].convert("RGB")
        objs = item["objects"]

        # COCO [x, y, w, h] -> [x1, y1, x2, y2]
        boxes = torch.tensor(objs["bbox"], dtype=torch.float32).reshape(-1, 4)
        boxes[:, 2:] += boxes[:, :2]

        target = {
            "boxes": tv_tensors.BoundingBoxes(boxes, format="XYXY",
                                              canvas_size=(img.height, img.width)),
            "labels": torch.tensor([ID2LABEL[c] for c in objs["category"]], dtype=torch.int64),
            "image_id": item["image_id"],
        }

        if self.transforms is not None:
            img, target = self.transforms(img, target)
        return img, target


# Sin cambios de tono/saturación: el color distingue varias clases (blue, pink, red, white)
# La normalización y el redimensionado los hace el propio modelo (GeneralizedRCNNTransform)
train_tf = v2.Compose([
    v2.ToImage(),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.ToDtype(torch.float32, scale=True),
    v2.SanitizeBoundingBoxes(),  # elimina cajas degeneradas tras las transformaciones
])
eval_tf = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

train_ds = PillsDataset(train_dataset, train_tf)
valid_ds = PillsDataset(valid_dataset, eval_tf)
test_ds  = PillsDataset(test_dataset,  eval_tf)
print(f"train: {len(train_ds)} | valid: {len(valid_ds)} | test: {len(test_ds)}")


def collate_fn(batch):
    # Cada imagen tiene un número distinto de cajas -> listas en vez de tensores apilados
    return tuple(zip(*batch))


BATCH_SIZE = 4
# num_workers=0: en Windows + Jupyter los workers no pueden importar clases definidas en el notebook
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)

# Comprobación rápida de un batch
images, targets = next(iter(train_loader))
print(images[0].shape, images[0].dtype, targets[0]["boxes"], targets[0]["labels"])

train: 315 | valid: 45 | test: 89
torch.Size([3, 640, 640]) torch.float32 BoundingBoxes([[241.0000, 261.0000, 369.5000, 408.0000]], format=BoundingBoxFormat.XYXY, canvas_size=(640, 640), clamping_mode=soft) tensor([4])
